# Demo — Árvore de seções: o modelo navega pelo sumário até a seção certa

**Objetivo:** responder perguntas sobre um documento estruturado **sem vetor e sem índice de busca**. Na ingestão, o modelo escreve um sumário hierárquico da norma; na consulta, lê só esse sumário, escolhe a seção e abre o texto dela. É a linha **Árvore (PageIndex)** do slide 18 e a linha **Árvore de seções** do slide 19 ("RAGless: recuperar sem banco vetorial, e quando isso cabe").

1. **Ingestão, parte determinística** → os títulos numerados da norma viram uma árvore de seções, sem modelo e sem custo;
2. **Ingestão, parte com LLM** → uma chamada escreve o resumo de cada seção: o sumário hierárquico;
3. **Navegar e abrir a seção certa** → o modelo lê só o sumário, escolhe a seção e responde citando a seção aberta;
4. **Quanto custou** → sumário + seção aberta contra o documento inteiro, com duas chamadas por pergunta.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

O documento é um só: a norma de segurança da informação (`acervo/ti/norma-de-seguranca-da-informacao.md`), com 8 seções `## ` e 22 subseções `### `, todas numeradas. O slide indica a árvore para documento longo e estruturado (contrato, norma, relatório). Para caber na aula, esta norma é curta, com cerca de 1.500 tokens, e a Execução 4 mostra quando a técnica se paga.

**Se o slide 18 ainda disser “Sem LLM na consulta” nesta linha**, leia pela descrição da própria linha: na consulta, o LLM navega pelo sumário (aqui, duas chamadas por pergunta). O que sai é o vetor, como diz o slide 19; o rótulo foi trocado com o de **Respostas prontas**.

## 0. Preparação

Este notebook usa só o **Azure OpenAI** (Azure OpenAI v1 API, cliente Python `OpenAI`). Não há embedding, índice nem AI Search.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT`, no formato `https://<recurso>.openai.azure.com`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat). Ele precisa aceitar **structured outputs**, porque o sumário e a navegação voltam num formato fixo; o `gpt-5-mini` sugerido no `.env.example` aceita.

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "pydantic>=2.7" "tiktoken>=0.7"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Acervo:", PASTA_ACERVO)

---

# Execução 1 — Ingestão, parte determinística: a árvore de seções

A norma já traz a própria estrutura: títulos numerados como `## 4.` e `### 4.2`. Algumas linhas de código transformam esses títulos numa **árvore**: cada nó tem o `id` (a numeração, como `"4"` ou `"4.2"`), o título, o nível, o texto próprio e os filhos.

Nenhum modelo é chamado aqui. A mesma norma gera sempre a mesma árvore, sem custo.

In [ ]:
import re

import tiktoken

DOCUMENTO = (PASTA_ACERVO / "ti" / "norma-de-seguranca-da-informacao.md").read_text(encoding="utf-8")

# Régua de tamanho: o tokenizador o200k_base, dos modelos GPT mais recentes.
codificador = tiktoken.get_encoding("o200k_base")


def contar_tokens(texto: str) -> int:
    return len(codificador.encode(texto))


def montar_arvore(texto: str) -> list[dict]:
    """Transforma os títulos numerados ("## 4.", "### 4.2") numa árvore; cada nó guarda o próprio texto."""
    raizes, pilha = [], []
    # re.split devolve [cabeçalho, marcas, número, título, texto, marcas, número, título, texto, ...]
    partes = re.split(r"(?m)^(#{2,6}) (\d+(?:\.\d+)*)\.? (.+)$", texto)
    for i in range(1, len(partes), 4):
        marcas, numero, titulo, texto_proprio = partes[i:i + 4]
        no = {"id": numero, "titulo": titulo.strip(), "nivel": len(marcas), "texto": texto_proprio.strip(), "filhos": []}
        # O pai é o último nó aberto com nível menor que o deste.
        while pilha and pilha[-1]["nivel"] >= no["nivel"]:
            pilha.pop()
        (pilha[-1]["filhos"] if pilha else raizes).append(no)
        pilha.append(no)
    return raizes


def percorrer(nos: list[dict]):
    """Todos os nós da árvore, na ordem do documento."""
    for no in nos:
        yield no
        yield from percorrer(no["filhos"])


arvore = montar_arvore(DOCUMENTO)
NOS = {no["id"]: no for no in percorrer(arvore)}

print(f"{len(arvore)} seções e {len(NOS) - len(arvore)} subseções; o documento inteiro tem {contar_tokens(DOCUMENTO)} tokens.\n")
for no in percorrer(arvore):
    recuo = "    " * (no["nivel"] - 2)
    print(f"{recuo}{no['id']}  {no['titulo']}  ({contar_tokens(no['texto'])} tokens de texto próprio)")

### O que observar

- A árvore saiu do próprio documento: os títulos numerados já são um sumário, só que sem conteúdo.
- As seções `##` aparecem com 0 tokens de texto próprio: o conteúdo está nas subseções `###`. Por isso, abrir a seção 4 é abrir 4.1, 4.2 e 4.3.
- Esta parte depende de títulos confiáveis. Num contrato em PDF, eles viriam da extração: o modelo Layout do Document Intelligence devolve o documento em Markdown, com seções e subseções (no AI Search, é a Document Layout skill do slide 12).

---

# Execução 2 — Ingestão, parte com LLM: o sumário hierárquico

Os títulos dizem onde fica cada assunto, mas não o que a seção decide: "Como reportar" não diz em quanto tempo. Por isso, **uma** chamada ao modelo lê a norma inteira e escreve um resumo de uma frase para cada nó da árvore.

- A resposta volta no formato das classes `ResumoSecao` e `Sumario` (**structured outputs**): o modelo é obrigado a seguir o esquema, e o código recebe objetos Python, não um texto para interpretar.
- O código manda junto a lista de `id`s da Execução 1: a estrutura vem do código; o modelo só escreve os resumos.
- A árvore com os resumos fica salva em `arvore.json`, ao lado do notebook. Rodar de novo lê o arquivo e não chama o modelo. **Mudou a norma? Apague o `arvore.json`** para gerar o sumário de novo.

### O que esperar e por que pode variar

- A redação dos resumos muda a cada geração. O que importa é que cada um diga o que a seção decide: regra, prazo, exceção.
- O esquema garante o formato, não a cobertura: se o modelo pular ou inventar um `id`, o código avisa.

In [ ]:
import json

from pydantic import BaseModel


class ResumoSecao(BaseModel):
    id: str
    resumo: str


class Sumario(BaseModel):
    secoes: list[ResumoSecao]


INSTRUCOES_SUMARIO = """
Você vai escrever o sumário de um documento. Quem vai ler o sumário é outra chamada do modelo,
que não verá o texto e precisará decidir, só pelo sumário, qual seção abrir para responder a uma pergunta.
Para cada id da lista, escreva em português um resumo de uma frase curta, de até 25 palavras,
que diga o que a seção estabelece (regras, prazos, valores e exceções), com as palavras que alguém
usaria para procurar o assunto. Numa seção que só agrupa subseções, resuma o conjunto.
Use exatamente os ids da lista, sem acrescentar nem pular nenhum.
""".strip()

ARQUIVO_ARVORE = Path.cwd() / "arvore.json"  # fica ao lado do notebook

if ARQUIVO_ARVORE.exists():
    salvos = json.loads(ARQUIVO_ARVORE.read_text(encoding="utf-8"))
    resumos = {no["id"]: no["resumo"] for no in percorrer(salvos)}
    print("Sumário lido de", ARQUIVO_ARVORE, "(nenhuma chamada ao modelo).")
else:
    resposta = client.responses.parse(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_SUMARIO,
        input=f"IDS: {', '.join(NOS)}\n\nDOCUMENTO:\n{DOCUMENTO}",
        text_format=Sumario,
        store=False,
    )
    if resposta.output_parsed is None:
        raise RuntimeError("O modelo não devolveu o sumário no formato pedido. Rode a célula de novo.")
    resumos = {s.id: s.resumo for s in resposta.output_parsed.secoes}
    print(f"Sumário gerado em uma chamada: {resposta.usage.input_tokens} tokens de entrada "
          f"e {resposta.usage.output_tokens} de saída.")

# A estrutura e o texto vêm sempre do documento; do modelo (ou do arquivo) vêm só os resumos.
for no in percorrer(arvore):
    no["resumo"] = resumos.get(no["id"], "")
if not ARQUIVO_ARVORE.exists():  # só na geração: rodar de novo não regrava o arquivo
    ARQUIVO_ARVORE.write_text(json.dumps(arvore, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Árvore com os resumos salva em", ARQUIVO_ARVORE)

sem_resumo = [i for i, no in NOS.items() if not no["resumo"]]
inventados = sorted(set(resumos) - set(NOS))
if sem_resumo or inventados:
    print("Atenção! Seções sem resumo:", sem_resumo, "| ids que não existem na árvore:", inventados)
    print("Apague o arvore.json e rode esta célula de novo.")

In [ ]:
# O sumário hierárquico, uma linha por seção, com recuo por nível. É tudo o que o modelo vê na consulta para escolher a seção.
SUMARIO = "\n".join(
    f"{'  ' * (no['nivel'] - 2)}{no['id']} {no['titulo']}: {no['resumo']}" for no in percorrer(arvore)
)

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro
display(pd.DataFrame([{"id": no["id"], "título": no["titulo"], "resumo": no["resumo"]} for no in percorrer(arvore)]))
print(f"Sumário: {contar_tokens(SUMARIO)} tokens | documento inteiro: {contar_tokens(DOCUMENTO)} tokens")

### O que observar

- O sumário é o "índice" desta abordagem, e é texto: dá para ler, revisar e versionar (abra o `arvore.json`). Num índice vetorial, o equivalente seria uma lista de números.
- Leia a tabela como o modelo vai lê-la na consulta: ela é **tudo** o que ele vê para escolher a seção. Um resumo vago, como "trata de incidentes", leva à navegação errada.
- O custo de LLM da ingestão é por documento, não por pergunta. Mas ele volta sempre que a norma muda.

### Fala sugerida

> “O sumário faz o papel do índice. Só que é texto: qualquer pessoa da área consegue ler e dizer se está certo.”

---

# Execução 3 — Navegar e abrir a seção certa

Cada pergunta passa por duas chamadas ao modelo, com o código no meio:

1. `navegar`: o modelo recebe **só o sumário** e a pergunta e devolve 1 ou 2 `id`s de seção, com uma justificativa (classe `Navegacao`, de novo com structured outputs);
2. `ids_para_abrir`: o código confere os `id`s (só os que existem na árvore, sem repetir, no máximo dois); em seguida, `abrir` pega o texto completo das seções escolhidas. Uma seção `##` leva junto as subseções;
3. `responder`: o modelo recebe **só as seções abertas** e responde citando `[seção X.Y]`.

São três perguntas, cada uma com a seção que a responde: 4.2 (ferramentas de IA generativa), 6.1 (como reportar um incidente) e 7.2 (localização dos dados). Esse gabarito fica no código, para a tabela; o modelo não o recebe.

### O que esperar e por que pode variar

- O modelo tende a escolher a subseção certa. Pode também abrir uma segunda seção (por exemplo, a 2.3 "Confidencial" junto com a 4.2) ou o capítulo inteiro ("4" em vez de "4.2"). Não é erro: a resposta recebe mais texto, e a tabela mostra. Se ele escolher o capítulo e uma subseção dele, o código abre só o capítulo, que já traz a subseção.
- Se ele escolher a seção errada, a resposta sai errada ou diz que não há evidência. A diferença para a busca vetorial é que o caminho e o motivo ficam escritos na tabela.

In [ ]:
class Navegacao(BaseModel):
    secoes: list[str]
    justificativa: str


INSTRUCOES_NAVEGACAO = """
Você recebe o sumário hierárquico de um documento e uma pergunta. Você não tem acesso ao texto das seções.
Escolha a seção, ou no máximo duas, cujo texto completo responde à pergunta.
Prefira a subseção mais específica (um id como "X.Y") ao capítulo inteiro ("X").
Devolva os ids exatamente como aparecem no sumário e justifique a escolha em uma frase, em português.
""".strip()

INSTRUCOES_RESPOSTA = """
Responda em português, exclusivamente com base nas seções fornecidas.
Cite a seção de cada afirmação no formato [seção X.Y].
Se as seções não trouxerem a informação, responda:
'Não há evidência suficiente nas seções abertas.'
""".strip()


def navegar(pergunta: str):
    """1ª chamada: o modelo lê só o sumário e escolhe as seções."""
    resposta = client.responses.parse(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_NAVEGACAO,
        input=f"SUMÁRIO:\n{SUMARIO}\n\nPERGUNTA:\n{pergunta}",  # o que muda, a pergunta, vai no fim
        text_format=Navegacao,
        store=False,
    )
    if resposta.output_parsed is None:
        raise RuntimeError("O modelo não devolveu a navegação no formato pedido. Rode a célula de novo.")
    return resposta


def ids_para_abrir(escolhidas: list[str]) -> list[str]:
    """Os ids que o código abre: só os da árvore ("Seção 3.2" vale "3.2"), sem repetir, no máximo dois."""
    ids = []
    for texto in escolhidas:
        numero = re.search(r"\d+(?:\.\d+)*", texto)
        if numero and numero.group() in NOS and numero.group() not in ids:
            ids.append(numero.group())
    # Uma subseção cujo capítulo também foi escolhido já vai junto com ele.
    ids = [i for i in ids if not any(i.startswith(outro + ".") for outro in ids)]
    return ids[:2]


def texto_completo(no: dict) -> str:
    """Título e texto da seção, seguidos dos das subseções."""
    partes = [f"[seção {no['id']}] {no['titulo']}", no["texto"], *(texto_completo(filho) for filho in no["filhos"])]
    return "\n".join(p for p in partes if p)


def abrir(ids: list[str]) -> str:
    """O código abre o texto completo das seções escolhidas."""
    return "\n\n".join(texto_completo(NOS[i]) for i in ids)


def responder(pergunta: str, secoes_abertas: str):
    """2ª chamada: o modelo responde só com as seções abertas."""
    return client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RESPOSTA,
        input=f"SEÇÕES ABERTAS:\n{secoes_abertas}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )


def caminho(id_secao: str) -> str:
    """O caminho na árvore até a seção. Pela numeração, o ancestral de "4.2" é "4"."""
    numeros = id_secao.split(".")
    ancestrais = [".".join(numeros[:i]) for i in range(1, len(numeros) + 1)]
    return " → ".join(f"{a} {NOS[a]['titulo']}" for a in ancestrais)

In [ ]:
PERGUNTAS_NORMA = {  # pergunta → seção que a responde (o gabarito desta demo)
    "Posso colar um relatório confidencial num chatbot gratuito para resumir?": "4.2",
    "Em quanto tempo preciso reportar um incidente de segurança?": "6.1",
    "Dados de clientes podem ficar num data center fora do Brasil?": "7.2",
}

resultados = []
for pergunta, esperada in PERGUNTAS_NORMA.items():
    navegacao = navegar(pergunta)
    escolhidas = navegacao.output_parsed.secoes  # o que o modelo devolveu, como veio
    ids = ids_para_abrir(escolhidas)
    secoes_abertas = abrir(ids)
    resposta = responder(pergunta, secoes_abertas)
    resultados.append({
        "pergunta": pergunta,
        "esperada": esperada,
        "escolhidas": escolhidas,
        "ids": ids,
        "justificativa": navegacao.output_parsed.justificativa,
        "secoes_abertas": secoes_abertas,
        "resposta": resposta.output_text,
        "uso": [navegacao.usage, resposta.usage],
    })

display(pd.DataFrame([
    {
        "pergunta": r["pergunta"],
        "seção esperada": r["esperada"],
        "o modelo escolheu": ", ".join(r["escolhidas"]) or "nada",
        "seções abertas (caminho na árvore)": "; ".join(caminho(i) for i in r["ids"]) or "nenhuma",
        "justificativa do modelo": r["justificativa"],
    }
    for r in resultados
]))

In [ ]:
for r in resultados:
    display(Markdown(f"### {r['pergunta']}"))
    display(Markdown(f"**Seções abertas:** {', '.join(r['ids']) or 'nenhuma'} (esperada: {r['esperada']})"))
    display(Markdown(r["resposta"]))

### O que observar

- Compare **seções abertas** com **seção esperada**: é a pergunta do slide 29 ("o trecho certo foi recuperado?"), só que aqui quem recupera é o modelo, e ele deixa o motivo por escrito. A coluna **o modelo escolheu** mostra o que ele devolveu, antes da conferência do código.
- Cada afirmação da resposta cita `[seção X.Y]`: a citação aponta para um lugar do documento que qualquer pessoa abre e confere.
- Na consulta, o modelo nunca leu a norma inteira: na 1ª chamada viu o sumário; na 2ª, só as seções abertas.

### Leitura arquitetural

- **Sem vetor e sem índice de busca:** nenhum embedding, nenhum AI Search. A recuperação é o modelo lendo um sumário, e o "índice" é o `arvore.json`.
- **A qualidade depende do sumário.** Resumo ruim leva à seção errada, e seção errada leva à resposta errada. O sumário passa a ser o que se mede e se revisa, como o chunking no RAG clássico.
- **Navegação errada é resposta errada, mas auditável:** o caminho e a justificativa ficam registrados. Dá para saber se o erro foi da navegação ou da geração.

---

# Execução 4 — Quanto custou

Não há índice para manter, mas cada pergunta tem custo: o sumário inteiro vai na 1ª chamada, toda vez, e as seções abertas vão na 2ª. A comparação é com a alternativa sem recuperação nenhuma: mandar o documento inteiro junto com a pergunta, numa chamada só (o contexto longo de `../01-contexto-longo`).

- **Texto enviado** é contado com o tokenizador `o200k_base`, sobre o sumário e as seções abertas, como foram para o prompt (sem as instruções e a pergunta).
- **Entrada e saída cobradas** vêm do `usage` das duas chamadas, como a API registrou. A saída inclui os tokens de raciocínio dos modelos de raciocínio, como os gpt-5: eles não aparecem no texto, mas são cobrados como saída.
- **Em cache** é a parte da entrada cobrada que o serviço leu do cache de prompt (`usage.input_tokens_details.cached_tokens`). Ela já está somada na entrada, mas é cobrada com desconto.

### O que esperar e por que pode variar

- A norma é curta, com cerca de 1.500 tokens, e o sumário pode ficar perto do tamanho dela. A economia de texto tende a ser pequena, e as duas chamadas podem custar mais do que uma chamada com o documento inteiro. Os números mudam com o modelo, com o tamanho dos resumos e com as seções abertas.
- O começo da 1ª chamada (instruções + sumário) é igual em todas as perguntas e tende a passar de 1.024 tokens, o mínimo do cache de prompt. Por isso, da 2ª pergunta em diante, a coluna **em cache** tende a mostrar parte da entrada lida do cache. A 1ª pergunta tende a vir com 0, porque é ela que grava esse começo (se o notebook rodou há poucos minutos, até ela pode vir do cache). A 2ª chamada muda a cada pergunta e é curta demais para o cache.
- O acerto não é garantido: o cache é temporário, e, se o modelo escrever resumos mais curtos, o começo pode ficar abaixo de 1.024 tokens. Aí a coluna fica em 0.
- Essas regras valem para os modelos do GPT-4o até a família GPT-5.5, como o `gpt-5-mini`. Da família GPT-5.6 em diante, o cache só é lido nos pontos de corte (*breakpoints*), e o padrão fica no fim da última mensagem, que aqui já inclui a pergunta: a coluna **em cache** pode ficar em 0 (veja `../01-contexto-longo`).

In [ ]:
tokens_documento = contar_tokens(DOCUMENTO)
tokens_sumario = contar_tokens(SUMARIO)
com_texto = sum(1 for no in NOS.values() if no["texto"])  # as subseções: as seções ## não têm texto próprio
print(f"Documento inteiro: {tokens_documento} tokens, ~{tokens_documento / com_texto:.0f} por subseção ({com_texto} subseções).")
print(f"Sumário, enviado em toda pergunta: {tokens_sumario} tokens, ~{tokens_sumario / len(NOS):.0f} por linha "
      f"({len(NOS)} linhas, uma por nó da árvore).\n")

linhas = []
for r in resultados:
    enviado = tokens_sumario + contar_tokens(r["secoes_abertas"])
    linhas.append({
        "pergunta": r["pergunta"],
        "seções abertas": ", ".join(r["ids"]) or "nenhuma",
        "texto enviado: sumário + seções (tokens)": enviado,
        "em relação ao documento inteiro": f"{enviado / tokens_documento:.0%}",
        "entrada cobrada (2 chamadas)": sum(u.input_tokens for u in r["uso"]),
        # Parte da entrada lida do cache de prompt; se o serviço não mandar o detalhamento, conta como 0.
        "em cache (2 chamadas)": sum(
            (u.input_tokens_details.cached_tokens if u.input_tokens_details else 0) or 0 for u in r["uso"]
        ),
        "saída cobrada (2 chamadas)": sum(u.output_tokens for u in r["uso"]),
    })

display(pd.DataFrame(linhas))

### Leitura arquitetural

- **O custo migra para a consulta** (slide 19): não há índice para manter, mas toda pergunta paga o sumário e duas chamadas, e espera as duas. A ingestão foi uma chamada com o documento inteiro, uma vez só.
- **A árvore se paga em documento longo.** O sumário cresce com o número de seções; o documento cresce com o tamanho delas. Numa norma curta como esta, a diferença é pequena. Num contrato com seções de duas páginas, cada seção passa de mil tokens de texto, e o resumo dela continua com algumas dezenas: o sumário vira uma fração pequena do documento.
- **O sumário é o começo fixo da 1ª chamada**, por isso vem antes da pergunta. Aqui, instruções + sumário tendem a passar de 1.024 tokens, o mínimo do cache de prompt do Azure OpenAI: da 2ª pergunta em diante, esse começo pode ser lido do cache (coluna **em cache**) e sai com desconto. Isso vale do GPT-4o até a família GPT-5.5; da GPT-5.6 em diante, reaproveitar o sumário entre perguntas pede um ponto de corte logo depois dele. O notebook `../01-contexto-longo` mostra o cache em detalhe.
- **Não serve para milhares de documentos**, nem para acervo volátil ou com permissão por usuário: o sumário de tudo não cabe no prompt, e a árvore não sabe quem perguntou. Aí o índice volta a valer (slides 18 e 19).

### Fala sugerida

> “A árvore troca infraestrutura por consulta: não tem índice para manter, mas toda pergunta paga o sumário e duas chamadas. Em documento longo, a conta fecha; em documento curto, mande o documento inteiro.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os títulos numerados viram uma árvore, sem modelo e sem custo | A estrutura do documento faz o papel do índice, desde que os títulos sejam confiáveis |
| 2 | Uma chamada escreve o sumário hierárquico, salvo em `arvore.json` | O LLM vai para a ingestão; o sumário é gerado de novo quando o documento muda |
| 3 | O modelo navega só pelo sumário, abre a seção e cita `[seção X.Y]` | Recuperação auditável: o caminho e o motivo ficam escritos |
| 4 | Duas chamadas por pergunta, e o sumário vai em todas | Cabe em documento longo e estruturado; acervo grande pede o índice de volta |

## Frase de fechamento

> **Na ingestão, o LLM gera um sumário hierárquico; na consulta, navega por ele e abre a seção certa**

---

# Referências oficiais

- Azure OpenAI — structured outputs, o formato fixo do sumário e da navegação:  
  https://learn.microsoft.com/azure/foundry/openai/how-to/structured-outputs

- Azure OpenAI — modelos de raciocínio e os tokens de raciocínio, cobrados como saída:  
  https://learn.microsoft.com/azure/foundry/openai/how-to/reasoning

- Azure OpenAI — cache de prompt (prompt caching):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/prompt-caching

- Document Intelligence — modelo Layout, com seções, subseções e saída em Markdown:  
  https://learn.microsoft.com/azure/ai-services/document-intelligence/prebuilt/layout